# RAG UPV Avanzado - Con Calculadora y Visualizaciones

Sistema RAG mejorado que accede a los datos abiertos de la UPV con capacidades avanzadas:
- 📊 Análisis de datasets de UPV Transparent
- 🧮 Calculadora matemática y estadística
- 💻 Ejecución de código Python
- 📈 Generación de visualizaciones
- 🔍 Búsqueda semántica con FAISS
- 🤖 Agente inteligente con Groq (Llama 3.1)

## 1. Instalación de Dependencias

In [ ]:
!pip install -q groq sentence-transformers faiss-cpu pandas matplotlib seaborn plotly numpy scipy scikit-learn requests openpyxl

## 2. Imports y Configuración

In [ ]:
import requests
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
from groq import Groq
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from sklearn.linear_model import LinearRegression
import json
import os
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualizaciones
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# API Key de Groq - usa variable de entorno o ingresa la tuya aquí
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "TU_API_KEY_AQUI")
if GROQ_API_KEY == "TU_API_KEY_AQUI":
    print("⚠️  IMPORTANTE: Configura tu API key de Groq")
    print("   Opción 1: export GROQ_API_KEY='tu-api-key'")
    print("   Opción 2: Reemplaza 'TU_API_KEY_AQUI' en el código")
    print("   Obtén tu API key gratis en: https://console.groq.com/")
else:
    client = Groq(api_key=GROQ_API_KEY)
    print("✅ Configuración completada")

## 3. Cargar Datasets de UPV Transparent

In [ ]:
# Función para obtener metadatos de todos los datasets
def get_datasets():
    """Obtiene metadatos de todos los datasets de UPV Transparent"""
    url = "https://upvtransparent.upv.es/api/3/action/package_list"
    response = requests.get(url)
    packages = response.json()['result']

    metadata = []
    print(f"📦 Cargando {len(packages)} datasets...")
    
    for i, pkg in enumerate(packages):
        try:
            show_url = f"https://upvtransparent.upv.es/api/3/action/package_show?id={pkg}"
            resp = requests.get(show_url)
            data = resp.json()['result']
            title = data['title']
            notes = data.get('notes', '')
            resources = [(r['name'], r['url'], r['format'].lower()) for r in data.get('resources', [])]
            metadata.append({
                'id': pkg,
                'title': title,
                'notes': notes,
                'resources': resources
            })
            
            if (i + 1) % 10 == 0:
                print(f"   Procesados {i + 1}/{len(packages)} datasets...")
        except Exception as e:
            print(f"   ⚠️  Error en dataset {pkg}: {e}")
            continue

    return metadata

# Cargar metadatos
print("🔍 Cargando datasets de UPV Transparent...")
metadata = get_datasets()
print(f"\n✅ {len(metadata)} datasets cargados exitosamente")

## 4. Crear Índice de Búsqueda Semántica

In [ ]:
# Modelo de embeddings local (gratuito)
print("📚 Cargando modelo de embeddings...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Crear embeddings de títulos + notas
print("🔄 Generando embeddings...")
texts = [m['title'] + "\n" + m['notes'] for m in metadata]
embeddings = model.encode(texts, show_progress_bar=True)

# Crear índice FAISS
print("🗂️  Creando índice FAISS...")
d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(np.array(embeddings))

print(f"\n✅ Índice creado con {index.ntotal} vectores de dimensión {d}")

## 5. Definir Herramientas para el Agente

In [ ]:
# Cache para datasets cargados
dataset_cache = {}

def load_dataset_data(dataset_id: str, max_rows: int = 1000) -> pd.DataFrame:
    """Carga datos de un dataset específico"""
    if dataset_id in dataset_cache:
        return dataset_cache[dataset_id]
    
    # Buscar dataset
    dataset = next((m for m in metadata if m['id'] == dataset_id), None)
    if not dataset or not dataset['resources']:
        return None
    
    # Intentar cargar el primer resource
    for name, url, fmt in dataset['resources']:
        try:
            if 'csv' in fmt:
                df = pd.read_csv(url, nrows=max_rows)
            elif 'xls' in fmt:
                df = pd.read_excel(url, nrows=max_rows)
            else:
                continue
            
            dataset_cache[dataset_id] = df
            return df
        except Exception as e:
            continue
    
    return None

# Definir herramientas para Groq
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_upv_datasets",
            "description": "Busca datasets relevantes de UPV Transparent basado en una consulta. Retorna títulos, descripciones y enlaces a los datos.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Consulta de búsqueda para encontrar datasets relevantes"
                    },
                    "k": {
                        "type": "integer",
                        "description": "Número de resultados a retornar (default: 3)"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "load_dataset",
            "description": "Carga datos reales de un dataset específico de UPV. Retorna las primeras filas y estructura.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dataset_id": {
                        "type": "string",
                        "description": "ID del dataset a cargar (obtener de search_upv_datasets)"
                    },
                    "rows": {
                        "type": "integer",
                        "description": "Número de filas a mostrar (default: 5)"
                    }
                },
                "required": ["dataset_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Ejecuta cálculos matemáticos y estadísticos complejos. Soporta numpy, scipy, pandas.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Expresión matemática Python. Ej: 'np.mean([1,2,3]) + np.sqrt(16)'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_dataset",
            "description": "Realiza análisis estadístico de un dataset: estadísticas descriptivas, correlaciones, etc.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dataset_id": {
                        "type": "string",
                        "description": "ID del dataset a analizar"
                    },
                    "analysis_type": {
                        "type": "string",
                        "enum": ["descriptive", "correlation", "summary"],
                        "description": "Tipo de análisis a realizar"
                    },
                    "columns": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Columnas específicas a analizar (opcional)"
                    }
                },
                "required": ["dataset_id", "analysis_type"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_visualization",
            "description": "Crea visualizaciones de datos de un dataset: gráficos de línea, barras, dispersión, histogramas, etc.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dataset_id": {
                        "type": "string",
                        "description": "ID del dataset a visualizar"
                    },
                    "chart_type": {
                        "type": "string",
                        "enum": ["line", "bar", "scatter", "histogram", "box", "heatmap"],
                        "description": "Tipo de gráfico"
                    },
                    "x_column": {
                        "type": "string",
                        "description": "Columna para eje X"
                    },
                    "y_column": {
                        "type": "string",
                        "description": "Columna para eje Y (requerido para algunos tipos)"
                    },
                    "title": {
                        "type": "string",
                        "description": "Título del gráfico"
                    }
                },
                "required": ["dataset_id", "chart_type"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "execute_python",
            "description": "Ejecuta código Python personalizado para análisis avanzado. Tiene acceso a pandas (pd), numpy (np), matplotlib (plt).",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "Código Python a ejecutar. Usa 'df' para acceder al último dataset cargado. Asigna resultado a 'result'."
                    }
                },
                "required": ["code"]
            }
        }
    }
]

print("✅ Herramientas definidas:")
for tool in TOOLS:
    print(f"   - {tool['function']['name']}")

## 6. Implementar Funciones de las Herramientas

In [ ]:
# Variable global para el último dataset cargado
last_loaded_df = None

def search_upv_datasets(query: str, k: int = 3) -> str:
    """Busca datasets relevantes"""
    q_emb = model.encode([query])
    _, indices = index.search(q_emb, k)
    
    results = []
    for i in indices[0]:
        m = metadata[i]
        results.append({
            'id': m['id'],
            'title': m['title'],
            'description': m['notes'][:200] + '...' if len(m['notes']) > 200 else m['notes'],
            'resources': len(m['resources'])
        })
    
    return json.dumps(results, ensure_ascii=False, indent=2)

def load_dataset(dataset_id: str, rows: int = 5) -> str:
    """Carga datos de un dataset"""
    global last_loaded_df
    
    df = load_dataset_data(dataset_id, max_rows=1000)
    if df is None:
        return "Error: No se pudo cargar el dataset"
    
    last_loaded_df = df
    
    info = {
        'shape': df.shape,
        'columns': list(df.columns),
        'dtypes': {col: str(dtype) for col, dtype in df.dtypes.items()},
        'sample_data': df.head(rows).to_dict('records')
    }
    
    return json.dumps(info, ensure_ascii=False, indent=2)

def calculator(expression: str) -> str:
    """Evalúa expresiones matemáticas"""
    try:
        safe_dict = {
            'np': np,
            'pd': pd,
            'stats': stats,
            'sqrt': np.sqrt,
            'sin': np.sin,
            'cos': np.cos,
            'log': np.log,
            'exp': np.exp,
            'mean': np.mean,
            'median': np.median,
            'std': np.std,
            'sum': np.sum,
            'pi': np.pi,
            'e': np.e
        }
        result = eval(expression, {"__builtins__": {}}, safe_dict)
        return f"Resultado: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

def analyze_dataset(dataset_id: str, analysis_type: str, columns: list = None) -> str:
    """Analiza un dataset"""
    df = load_dataset_data(dataset_id)
    if df is None:
        return "Error: No se pudo cargar el dataset"
    
    try:
        if analysis_type == "descriptive":
            if columns:
                df = df[columns]
            return df.describe().to_string()
        
        elif analysis_type == "correlation":
            numeric_df = df.select_dtypes(include=[np.number])
            if columns:
                numeric_df = numeric_df[columns]
            return numeric_df.corr().to_string()
        
        elif analysis_type == "summary":
            info = {
                'shape': df.shape,
                'null_counts': df.isnull().sum().to_dict(),
                'dtypes': {col: str(dtype) for col, dtype in df.dtypes.items()}
            }
            return json.dumps(info, indent=2)
        
        return "Tipo de análisis no soportado"
    except Exception as e:
        return f"Error: {str(e)}"

def create_visualization(dataset_id: str, chart_type: str, x_column: str = None, 
                        y_column: str = None, title: str = "") -> str:
    """Crea visualizaciones"""
    df = load_dataset_data(dataset_id)
    if df is None:
        return "Error: No se pudo cargar el dataset"
    
    try:
        plt.figure(figsize=(12, 6))
        
        if chart_type == "line" and x_column and y_column:
            plt.plot(df[x_column], df[y_column])
            plt.xlabel(x_column)
            plt.ylabel(y_column)
        
        elif chart_type == "bar" and x_column and y_column:
            df_plot = df.groupby(x_column)[y_column].sum().head(20)
            plt.bar(range(len(df_plot)), df_plot.values)
            plt.xticks(range(len(df_plot)), df_plot.index, rotation=45, ha='right')
            plt.xlabel(x_column)
            plt.ylabel(y_column)
        
        elif chart_type == "scatter" and x_column and y_column:
            plt.scatter(df[x_column], df[y_column], alpha=0.6)
            plt.xlabel(x_column)
            plt.ylabel(y_column)
        
        elif chart_type == "histogram" and x_column:
            plt.hist(df[x_column].dropna(), bins=30, edgecolor='black')
            plt.xlabel(x_column)
            plt.ylabel('Frecuencia')
        
        elif chart_type == "box":
            numeric_cols = df.select_dtypes(include=[np.number]).columns[:5]
            df[numeric_cols].boxplot()
            plt.xticks(rotation=45, ha='right')
        
        elif chart_type == "heatmap":
            numeric_df = df.select_dtypes(include=[np.number])
            sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', center=0)
        
        if title:
            plt.title(title)
        
        plt.tight_layout()
        plt.show()
        
        return f"✅ Gráfico '{chart_type}' creado exitosamente"
    except Exception as e:
        return f"Error: {str(e)}"

def execute_python(code: str) -> str:
    """Ejecuta código Python"""
    global last_loaded_df
    
    try:
        local_vars = {
            'pd': pd,
            'np': np,
            'plt': plt,
            'sns': sns,
            'df': last_loaded_df,
            'result': None
        }
        exec(code, {"__builtins__": __builtins__}, local_vars)
        return f"Resultado: {local_vars.get('result', 'Código ejecutado exitosamente')}"
    except Exception as e:
        return f"Error: {str(e)}"

# Mapeo de funciones
TOOL_FUNCTIONS = {
    "search_upv_datasets": search_upv_datasets,
    "load_dataset": load_dataset,
    "calculator": calculator,
    "analyze_dataset": analyze_dataset,
    "create_visualization": create_visualization,
    "execute_python": execute_python
}

print("✅ Funciones implementadas")

## 7. Agente RAG con Groq

In [ ]:
class UPVRAGAgent:
    """Agente RAG para datos UPV con capacidades avanzadas"""
    
    def __init__(self, model="llama-3.1-70b-versatile", max_iterations=10):
        self.model = model
        self.max_iterations = max_iterations
        self.conversation_history = []
    
    def process_tool_call(self, tool_name: str, tool_input: dict) -> str:
        """Procesa una llamada a herramienta"""
        print(f"\n🔧 Ejecutando: {tool_name}")
        print(f"   Parámetros: {json.dumps(tool_input, ensure_ascii=False, indent=2)}")
        
        if tool_name in TOOL_FUNCTIONS:
            result = TOOL_FUNCTIONS[tool_name](**tool_input)
            print(f"   ✅ Completado")
            return result
        else:
            return f"Error: herramienta '{tool_name}' no encontrada"
    
    def query(self, question: str) -> str:
        """Procesa una consulta del usuario"""
        print(f"\n{'='*80}")
        print(f"❓ Pregunta: {question}")
        print(f"{'='*80}")
        
        system_message = """Eres un asistente experto en análisis de datos de la UPV (Universitat Politècnica de València).

Capacidades:
- Buscar y analizar datasets de UPV Transparent
- Realizar cálculos matemáticos y estadísticos
- Crear visualizaciones de datos
- Ejecutar código Python para análisis personalizado

Proceso:
1. Busca datasets relevantes con search_upv_datasets
2. Carga los datos con load_dataset
3. Analiza o visualiza según la pregunta
4. Presenta resultados claros en español

Siempre responde en español y sé específico con los datos."""
        
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": question}
        ]
        
        for iteration in range(self.max_iterations):
            response = client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=TOOLS,
                tool_choice="auto",
                max_tokens=2048,
                temperature=0.3
            )
            
            response_message = response.choices[0].message
            
            if not response_message.tool_calls:
                final_response = response_message.content
                print(f"\n💡 Respuesta:\n{final_response}")
                return final_response
            
            messages.append(response_message)
            
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                function_response = self.process_tool_call(function_name, function_args)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": str(function_response)
                })
        
        return "Se alcanzó el límite de iteraciones"

# Inicializar agente
agent = UPVRAGAgent()
print("\n✅ Agente UPV RAG inicializado")
print("   Modelo: llama-3.1-70b-versatile")
print("   Datasets: {} disponibles".format(len(metadata)))

## 8. Ejemplos de Uso

### Ejemplo 1: Búsqueda Simple

In [ ]:
response = agent.query("¿Qué datos hay sobre estudiantes en la UPV?")

### Ejemplo 2: Análisis Estadístico

In [ ]:
response = agent.query(
    "Busca datos sobre presupuesto de la UPV, carga el dataset y muéstrame estadísticas descriptivas"
)

### Ejemplo 3: Visualización

In [ ]:
response = agent.query(
    "Encuentra datos de matriculaciones, carga el dataset y crea un gráfico mostrando la evolución"
)

### Ejemplo 4: Cálculos Personalizados

In [ ]:
response = agent.query(
    """Busca datos sobre investigación en la UPV.
    Carga el dataset y calcula:
    1. El promedio de proyectos por año
    2. La desviación estándar
    3. El crecimiento porcentual"""
)

### Ejemplo 5: Análisis Avanzado con Código

In [ ]:
response = agent.query(
    """Busca datos de personal de la UPV.
    Carga el dataset y ejecuta código Python para:
    - Filtrar solo docentes
    - Agrupar por departamento
    - Contar y mostrar los 5 departamentos con más personal"""
)

## 9. Modo Interactivo

In [ ]:
def interactive_mode():
    """Modo interactivo para hacer preguntas"""
    print("\n" + "="*80)
    print("🎓 MODO INTERACTIVO - RAG UPV Avanzado")
    print("="*80)
    print("\nPuedes preguntar sobre:")
    print("  - Búsqueda de datasets: '¿Qué datos hay sobre...?'")
    print("  - Análisis estadístico: 'Analiza los datos de...'")
    print("  - Visualizaciones: 'Crea un gráfico de...'")
    print("  - Cálculos: 'Calcula el promedio de...'")
    print("\nComandos:")
    print("  'salir' o 'exit' - Terminar")
    print("  'clear' - Limpiar cache de datasets")
    print("="*80 + "\n")
    
    while True:
        try:
            question = input("\n❓ Tu pregunta: ").strip()
            
            if not question:
                continue
            
            if question.lower() in ['salir', 'exit', 'quit']:
                print("\n👋 ¡Hasta luego!")
                break
            
            if question.lower() == 'clear':
                dataset_cache.clear()
                print("✅ Cache de datasets limpiado")
                continue
            
            agent.query(question)
            
        except KeyboardInterrupt:
            print("\n\n👋 ¡Hasta luego!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")

# Ejecutar modo interactivo
# Descomenta la siguiente línea para activar el modo interactivo
# interactive_mode()

## 10. Funciones de Acceso Rápido

In [ ]:
def quick_search(query: str):
    """Búsqueda rápida de datasets"""
    return agent.query(f"Busca datasets sobre: {query}")

def quick_analyze(dataset_name: str):
    """Análisis rápido de un dataset"""
    return agent.query(f"Busca el dataset '{dataset_name}', cárgalo y muéstrame estadísticas descriptivas")

def quick_viz(dataset_name: str, chart_type: str = "bar"):
    """Visualización rápida"""
    return agent.query(f"Busca el dataset '{dataset_name}', cárgalo y crea un gráfico tipo {chart_type}")

print("✅ Funciones de acceso rápido:")
print("   - quick_search(query): Buscar datasets")
print("   - quick_analyze(dataset_name): Análisis rápido")
print("   - quick_viz(dataset_name, chart_type): Visualización rápida")

## 11. Conclusión

Este notebook proporciona un sistema RAG avanzado para datos de la UPV con:

✅ **Acceso a datos abiertos** - API de UPV Transparent

✅ **Búsqueda semántica** - FAISS + SentenceTransformers

✅ **Cálculos y estadísticas** - NumPy, SciPy, Pandas

✅ **Visualizaciones** - Matplotlib, Seaborn, Plotly

✅ **Ejecución de código** - Python personalizado

✅ **Agente inteligente** - Groq (Llama 3.1) con function calling

### Preguntas de ejemplo:

- "¿Cuántos estudiantes hay matriculados en la UPV?"
- "Muéstrame un análisis del presupuesto de investigación"
- "Crea un gráfico de la evolución de publicaciones científicas"
- "Calcula la tasa de crecimiento anual de estudiantes de doctorado"
- "Compara los datos de diferentes campus"

### Próximos pasos:

1. Ejecuta `interactive_mode()` para modo conversacional
2. Prueba con diferentes tipos de análisis
3. Experimenta con visualizaciones personalizadas
4. Crea análisis combinando múltiples datasets